In [1]:
# generate data

import pandas as pd
import numpy as np

def generate_deepfm_ctr_data(n_samples=100000, seed=42):
    np.random.seed(seed)
    
    n_user = n_samples//100
    n_item = n_samples//1000

    user_data = pd.DataFrame(
        {
            "user_id": [i for i in range(n_user)],
            "gender" : np.random.choice(["M", "F"], n_user),
            "region" : np.random.choice(["서울", "경기", "경상", "전라", "충청", "강원"], size=n_user,
                                        p=[0.25, 0.25, 0.2, 0.15, 0.1, 0.05]),
            "age"    : np.random.randint(11, 50, n_user),
        }
    )
    
    item_data = pd.DataFrame(
        {
            "item_id"    : [i for i in range(n_item)],
            "category"   : np.random.choice(["fashion", "food", "electronics", "book", "sports"], n_item),
            "price"      : np.random.uniform(5, 500, n_item).round(2),
            "click_count": np.random.poisson(5, n_item)
        }
    )
    
    interaction_data = pd.DataFrame(
        {
            "user_id": np.random.randint(0, n_user, n_samples),
            "item_id": np.random.randint(0, n_item, n_samples),
            "device" : np.random.choice(["mobile", "pc", "tablet"], size=n_samples,
                                        p=[0.5, 0.4, 0.1]),
            "time"   : np.random.randint(0, 24, n_samples),
            "TARGET" : [1] * n_samples
        }
    )

    return user_data, item_data, interaction_data



In [2]:
# model_config.py
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# FEATURES / COLUMNS
SPARSE_FEATURES = ["gender", "region", "category"]
DENSE_FEATURES = ["age", "price", "click_count"]
USER_KEY_COL_NAME = "user_id"
ITEM_KEY_COL_NAME = "item_id"
TARGET_KEY_COLS = ["TARGET"]

# PREPROCESSING  
ENCODER = LabelEncoder
SCALER = MinMaxScaler
# SCALER = "HASH"

# HYPER PARAMS
BATCH_SIZE = 256
EPOCHS = 10
EMBEDDING_DIMS=4                  # 8, 16, 32, ...
OPTIMIZER = "adam"                # "rmsprop" ... See tf.keras.optimizers
LOSS = "binary_crossentropy"      # See tf.keras.losses.LOSS
METRICS = ["binary_crossentropy"] # See tf.keras.metrics

# OTHER
TEST_RATIO = 0.2
VALID_RATIO = 0.2
VERBOSE = 2

# INFERENCE
TOP_N = 10
SCORE_COL_NAME = "score"


In [3]:

def load_data(sparse_features:list[str]=SPARSE_FEATURES,
              dense_features:list[str]=DENSE_FEATURES):
    
    # 데이터 로딩
    user_data, item_data, interaction_data = generate_deepfm_ctr_data()
    
    # 범주형 피처와 수치형 피처
    sparse_features = sparse_features
    dense_features  = dense_features
    
    return user_data, item_data, interaction_data, sparse_features, dense_features


In [4]:
def data_preprocess(user_data, item_data, interaction_data, sparse_features, dense_features,
                    user_key_col_name:str=USER_KEY_COL_NAME, item_key_col_name:str=ITEM_KEY_COL_NAME,
                    encoder=ENCODER, scaler=SCALER):
    
    # 데이터 결합
    data = interaction_data.copy()
    data = data.merge(user_data, how="left", on=user_key_col_name)
    data = data.merge(item_data, how="left", on=item_key_col_name)
    
    # 결측치 처리 (NOTE:예시임, 추후 프로젝트에서는 알맞은 전처리 도입)
    data[sparse_features] = data[sparse_features].fillna("-1")
    data[dense_features]  = data[dense_features].fillna(0)
    
    # 전처리 : 범주형 피처 인코딩
    encoders = dict()
    for feat in sparse_features:
        lbe = encoder()
        data[feat] = lbe.fit_transform(data[feat])
        encoders[feat] = lbe
        
    # 전처리 : 수치형 피처 정규화 (유저, 아이템 각각)
    scalers = dict()
    
    dense_scaler = scaler(feature_range=(0, 1))
    user_dense_features = [col for col in user_data.columns if col in DENSE_FEATURES]
    data[user_dense_features] = dense_scaler.fit_transform(data[user_dense_features])
    scalers["user"] = dense_scaler
    
    dense_scaler = scaler(feature_range=(0, 1))
    item_dense_features = [col for col in item_data.columns if col in DENSE_FEATURES]
    data[item_dense_features] = dense_scaler.fit_transform(data[item_dense_features])
    scalers["item"] = dense_scaler
    
    return data, encoders, scalers


In [5]:
from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names

def generate_feature_columns(data, sparse_features:list[str]=SPARSE_FEATURES, dense_features:list[str]=DENSE_FEATURES):
    fixlen_feature_columns = [
        SparseFeat(feat, vocabulary_size=data[feat].max() + 1, embedding_dim=4)
        for i, feat in enumerate(sparse_features)
    ] + [
        DenseFeat(feat, 1)
        for feat in dense_features
    ]

    dnn_feature_columns = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns
    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)
    return feature_names, linear_feature_columns, dnn_feature_columns



In [ ]:
from sklearn.model_selection import train_test_split
from deepctr.models import DeepFM

def train_deepfm_model(data, feature_names, linear_feature_columns, dnn_feature_columns, target_key_cols=TARGET_KEY_COLS,
                       test_ratio:float=TEST_RATIO, valid_ratio:float=VALID_RATIO, batch_size:int=BATCH_SIZE, epochs:int=EPOCHS,
                       verbose=VERBOSE, optimizer:str=OPTIMIZER, loss:str=LOSS, metrics:list[str]=METRICS):
    
    # Train Test Split
    train, test = train_test_split(data, test_size=test_ratio)

    # Train, Test 데이터
    train_model_input = {name:train[name].values for name in feature_names}
    test_model_input = {name:test[name].values for name in feature_names}

    # DeepFM 모델 생성 및 컴파일
    model = DeepFM(linear_feature_columns,dnn_feature_columns,task='binary')
    model.compile(optimizer, loss, metrics=metrics, )

    # 훈련
    history = model.fit(train_model_input, train[target_key_cols].values,
                        batch_size=batch_size, epochs=epochs, verbose=verbose, validation_split=valid_ratio, )
    
    return model, test


In [8]:
user_data, item_data, interaction_data, sparse_features, dense_features = load_data(SPARSE_FEATURES, DENSE_FEATURES)
data, encoders, scalers = data_preprocess(user_data, item_data, interaction_data, SPARSE_FEATURES, DENSE_FEATURES)
feature_names, linear_feature_columns, dnn_feature_columns = generate_feature_columns(data, SPARSE_FEATURES, DENSE_FEATURES)
model, test, pred_ans = train_deepfm_model(data, feature_names, linear_feature_columns, dnn_feature_columns)


ValueError: `name` must be passed as a keyword argument. Received: add_weight('linear_kernel', ...). Use: add_weight(shape=..., name='linear_kernel').

In [ ]:
# Artifact 저장

# 유저 데이터, 아이템 데이터를 저장해둬, 추론시 재활용
for feat in [col for col in user_data.columns if col in SPARSE_FEATURES]:
    user_data[feat] = encoders[feat].transform(user_data[feat])
user_dense_features = [col for col in user_data if col in DENSE_FEATURES]
user_data[user_dense_features] = scalers["user"].transform(user_data[user_dense_features])

for feat in [col for col in item_data.columns if col in SPARSE_FEATURES]:
    item_data[feat] = encoders[feat].transform(item_data[feat])
item_dense_features = [col for col in item_data if col in DENSE_FEATURES]
item_data[item_dense_features] = scalers["item"].transform(item_data[item_dense_features])

# 하이퍼파라미터
hyper_params = {
    "batch_size" : BATCH_SIZE,
    "epochs" : EPOCHS
}

artifacts = {
    "model" : model,
    "encoders" : encoders,
    "scalers" : scalers,
    "sparse_features" : SPARSE_FEATURES,
    "dense_features" : DENSE_FEATURES,
    "feature_names" : feature_names,
    "processed_user_data" : user_data,
    "processed_item_data" : item_data,
    "hyper_params" : hyper_params
}


In [ ]:
def predict(user_id, artifacts):
    
    # 추론에 필요한 데이터 로딩
    user_data = artifacts["processed_user_data"].copy()
    infer_item_data = artifacts["processed_item_data"].copy()
    
    # 추론용 데이터셋 생성
    infer_user_data = user_data[user_data["user_id"] == user_id]
    infer_user_data["merge_col"] = 1    
    infer_item_data["merge_col"] = 1
    infer_data = infer_user_data.merge(infer_item_data, on="merge_col")
    infer_data = infer_data.drop(columns="merge_col")
    
    # 추론용 input data
    feature_names = artifacts["feature_names"]
    model_input = {feat:infer_data[feat].values for feat in feature_names}
    
    # 추론
    scores = model.predict(model_input, batch_size=artifacts["hyper_params"]["batch_size"])
    
    # item_id와 결합 및 추천 개수 슬라이싱
    result = pd.DataFrame({ITEM_KEY_COL_NAME:infer_data[ITEM_KEY_COL_NAME], SCORE_COL_NAME:[score[0] for score in scores]})
    result = result.sort_values(by=SCORE_COL_NAME, ascending=False)
    
    return result


In [9]:
data

,user_id,item_id,device,time,TARGET,gender,region,age,category,price,click_count
0,559,14,pc,6,1,0,2,0.473684,0,0.600137,0.416667
1,916,27,mobile,2,1,0,3,0.263158,0,0.007827,0.333333
2,890,83,mobile,4,1,1,3,0.210526,3,0.525552,0.666667
3,247,13,tablet,17,1,1,2,0.789474,3,0.673640,0.666667
4,789,89,mobile,11,1,1,3,0.868421,0,0.260080,0.250000
...,...,...,...,...,...,...,...,...,...,...,...
99995,242,55,mobile,12,1,0,4,0.631579,1,0.243157,0.500000
99996,87,15,pc,7,1,1,3,0.842105,3,0.351610,0.333333
99997,982,44,pc,6,1,0,2,0.184211,1,0.377256,0.416667
99998,479,46,pc,21,1,0,2,0.315789,4,0.326672,0.333333
